# StockSense FinBERT GPU training

Use a Colab runtime with **Runtime → Change runtime type → T4 GPU**. This notebook trains the existing project challenger on the frozen Financial PhraseBank split. It does not contain or request NewsAPI, Marketaux, or OpenRouter keys.

In [ ]:
# Upload a ZIP of the repository, or replace this cell with your private clone command.
from google.colab import files
from pathlib import Path
import shutil, zipfile
uploaded = files.upload()
archive = next(iter(uploaded))
with zipfile.ZipFile(archive) as zf:
    zf.extractall('/content')
candidates = [p for p in Path('/content').iterdir() if p.is_dir() and (p / 'main.py').exists()]
assert candidates, 'Upload a repository ZIP containing main.py.'
repo = candidates[0]
%cd $repo

In [ ]:
!pip install -q -r requirements.txt
import torch
assert torch.cuda.is_available(), 'CUDA is unavailable; select a GPU runtime before continuing.'
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
!python main.py train_finbert --profile full --device cuda --no-promote

In [ ]:
# Export the staged checkpoint and provenance for local review.
from pathlib import Path
import shutil, zipfile
staged = sorted(Path('models/experimental').glob('finbert-sentiment-staging-*'))[-1]
with zipfile.ZipFile('/content/stocksense-finbert-gpu-artifact.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for path in staged.rglob('*'):
        if path.is_file(): zf.write(path, path.relative_to(staged.parent))
    for path in Path('results/finbert-benchmark').glob('*'):
        if path.is_file(): zf.write(path, path.relative_to(Path('.')))
files.download('/content/stocksense-finbert-gpu-artifact.zip')